# Kitsune on Google Colab via Git Clone

This notebook installs dependencies, clones `Kitsune-py` from GitHub, mounts Google Drive, and runs Kitsune on a capture file.

Recommended training pattern:
- use the early benign traffic for training
- use later traffic for anomaly scoring


In [ ]:
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y tshark
!pip install -q scapy cython numpy scipy matplotlib
!tshark --version | head -n 1


In [ ]:
import os
import shutil
import subprocess

REPO_URL = 'https://github.com/ymirsky/Kitsune-py.git'
REPO_DIR = '/content/Kitsune-py'

if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
print('Cloned repo to', REPO_DIR)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
capture_path = '/content/drive/MyDrive/ARP_MitM_pcap.pcapng'

if not os.path.isfile(capture_path):
    raise FileNotFoundError(f'Capture not found: {capture_path}')

print('Using capture:', capture_path)
print('tshark path:', shutil.which('tshark'))


In [ ]:
import sys
import numpy as np

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

from Kitsune import Kitsune


In [ ]:
maxAE = 10
FMgrace = 5000
ADgrace = 50000
packet_limit = np.inf

K = Kitsune(capture_path, packet_limit, maxAE, FMgrace, ADgrace)


In [ ]:
rmse_scores = []
i = 0

while True:
    i += 1
    if i % 1000 == 0:
        print(f'Processed {i} packets')
    rmse = K.proc_next_packet()
    if rmse == -1:
        break
    rmse_scores.append(rmse)

print('Packets processed:', len(rmse_scores))


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.plot(rmse_scores)
plt.yscale('log')
plt.title('Kitsune RMSE Scores')
plt.xlabel('Packet index')
plt.ylabel('RMSE (log scale)')
plt.show()


## Notes

- The first `FMgrace + ADgrace` packets are used for learning.
- If your capture starts with attack traffic, do not treat those packets as clean training data.
- After the first run, Kitsune will save a `.tsv` next to the capture, which is faster to reuse.
- If you maintain your own fork, replace `REPO_URL` with your GitHub repository URL.
